In [1]:
import pandas as pd
import numpy as np
import pgeocode

rng = np.random.default_rng(seed=42)

from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

In [2]:
path_folder = "../data/processed/"
path_claim = "claims_with_imputed_severity.xlsx"

In [3]:
ds_claim_main = pd.read_excel(
    path_folder + path_claim,
    sheet_name=0,
    dtype={"Accident Zip": str}
)

In [4]:
ds_claim_main = ds_claim_main[["Claim Number","NOL Date","Division","CAT Severity Code","Accident Zip"]]

In [5]:
sla_table = pd.DataFrame({
    "CAT Severity Code": [1, 2, 3, 4, 5],
    "SLA Days": [28, 28, 21, 14, 7]
})

In [6]:
ds_claim_main = ds_claim_main.merge(
    sla_table,
    on="CAT Severity Code",
    how="left"
)

In [7]:
ds_claim_main["Due Date"] = (
    ds_claim_main["NOL Date"] +
    pd.to_timedelta(ds_claim_main["SLA Days"], unit="D")
)

In [8]:
ds_claim_main = ds_claim_main.replace(
    r"^\s*$",
    np.nan,
    regex=True
)

In [9]:
ds_claim_main.isna().sum().sort_values(ascending=False)

Accident Zip         3
NOL Date             0
Claim Number         0
Division             0
CAT Severity Code    0
SLA Days             0
Due Date             0
dtype: int64

In [10]:
ds_claim_main = ds_claim_main.dropna()

In [11]:
ds_claim_main

,Claim Number,NOL Date,Division,CAT Severity Code,Accident Zip,SLA Days,Due Date
0,25266097,2023-12-10,PI,3,35209,21,2023-12-31
1,86498344,2023-12-10,PI,5,35209,7,2023-12-17
2,11851998,2023-12-11,PI,5,35243,7,2023-12-18
3,15735237,2023-12-21,PI,2,35661,28,2024-01-18
4,50814161,2023-12-10,PI,3,30060,21,2023-12-31
...,...,...,...,...,...,...,...
1059,34364820,2023-12-11,PI,5,37042,7,2023-12-18
1060,27252372,2023-12-10,PI,4,23236,14,2023-12-24
1062,90198592,2023-12-11,BI,3,35226,21,2024-01-01
1063,34785007,2023-12-11,BI,3,37066,21,2024-01-01


In [12]:
handletype_table = pd.DataFrame({
    "CAT Severity Code": [1, 2, 3, 4, 5],
    "On-Site Proportion": [0, 0.5, 0.75, 1, 1]
})

In [13]:
ds_claim_main = ds_claim_main.merge(
    handletype_table,
    on="CAT Severity Code",
    how="left"
)

In [14]:
def assign_on_site(group):
    p = group["On-Site Proportion"].iloc[0]

    if pd.isna(p):
        group["On-Site Handling"] = False
        return group

    n = len(group)
    n_on_site = int(round(n * p))

    on_site_idx = rng.choice(
        group.index,
        size=n_on_site,
        replace=False
    )

    group["On-Site Handling"] = False
    group.loc[on_site_idx, "On-Site Handling"] = True

    return group

In [15]:
ds_claim_main = (
    ds_claim_main
    .groupby(["NOL Date", "CAT Severity Code"], group_keys=False)
    .apply(assign_on_site)
    .reset_index(drop=True)
)

/tmp/ipykernel_3466628/3637903904.py:4: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(assign_on_site)


In [16]:
ds_claim_main["Virtual Handling"] = (
    ds_claim_main["On-Site Handling"]
    .fillna(False)
    .eq(False)
)

In [17]:
ds_claim_main[["On-Site Handling", "Virtual Handling"]].value_counts()

On-Site Handling  Virtual Handling
True              False               848
False             True                214
Name: count, dtype: int64

In [18]:
def normalize_us_zip(x) -> str | None:
    if pd.isna(x):
        return None

    s = str(x).strip()
    if not s:
        return None

    s = s.split("-")[0].strip()

    if s.endswith(".0"):
        s = s[:-2]

    digits = "".join(ch for ch in s if ch.isdigit())

    if len(digits) == 4:
        digits = digits.zfill(5)

    if len(digits) >= 5:
        return digits[:5]

    return None

def add_lat_lon_from_zip(
    df: pd.DataFrame,
    zip_col: str = "Accident Zip",
    country: str = "US",
    lat_col: str = "Latitude",
    lon_col: str = "Longitude",
    keep_normalized_zip: bool = True,
) -> pd.DataFrame:

    out = df.copy()

    norm_col = f"{zip_col}__zip5"
    out[norm_col] = out[zip_col].apply(normalize_us_zip)

    zips = (
        out[norm_col]
        .dropna()
        .astype(str)
        .unique()
    )

    nomi = pgeocode.Nominatim(country)

    lookup = nomi.query_postal_code(pd.Series(zips))

    zip_to_lat = dict(zip(lookup["postal_code"].astype(str), lookup["latitude"]))
    zip_to_lon = dict(zip(lookup["postal_code"].astype(str), lookup["longitude"]))

    out[lat_col] = out[norm_col].map(zip_to_lat).astype(float)
    out[lon_col] = out[norm_col].map(zip_to_lon).astype(float)

    if not keep_normalized_zip:
        out.drop(columns=[norm_col], inplace=True)

    return out

ds_claim_main = ds_claim_main.copy()

ds_claim_main = add_lat_lon_from_zip(
    ds_claim_main,
    zip_col="Accident Zip",
    lat_col="Lat",
    lon_col="Lon",
    keep_normalized_zip=True
)
ds_claim_main.drop(columns=["Accident Zip__zip5","On-Site Proportion"], inplace=True)

In [19]:
ds_claim_main

,Claim Number,NOL Date,Division,CAT Severity Code,Accident Zip,SLA Days,Due Date,On-Site Handling,Virtual Handling,Lat,Lon
0,25266097,2023-12-10,PI,3,35209,21,2023-12-31,True,False,33.4653,-86.8082
1,86498344,2023-12-10,PI,5,35209,7,2023-12-17,True,False,33.4653,-86.8082
2,11851998,2023-12-11,PI,5,35243,7,2023-12-18,True,False,33.4459,-86.7502
3,15735237,2023-12-21,PI,2,35661,28,2024-01-18,False,True,34.7561,-87.6304
4,50814161,2023-12-10,PI,3,30060,21,2023-12-31,True,False,33.9382,-84.5403
...,...,...,...,...,...,...,...,...,...,...,...
1057,34364820,2023-12-11,PI,5,37042,7,2023-12-18,True,False,36.5853,-87.4186
1058,27252372,2023-12-10,PI,4,23236,14,2023-12-24,True,False,37.4782,-77.5854
1059,90198592,2023-12-11,BI,3,35226,21,2024-01-01,True,False,33.3967,-86.8346
1060,34785007,2023-12-11,BI,3,37066,21,2024-01-01,False,True,36.3834,-86.4512


In [20]:
ds_claim_main.to_excel(
    "../data/processed/ds_claim_optClean.xlsx",
    index=False
)

In [33]:
ds_claim_main[ds_claim_main["Division"]=="BI"].count()

Claim Number         55
NOL Date             55
Division             55
CAT Severity Code    55
Accident Zip         55
SLA Days             55
Due Date             55
On-Site Handling     55
Virtual Handling     55
Lat                  55
Lon                  55
dtype: int64